# Datasets processing

In [1]:
# Processing procedures of the data we are using 

# Adult Income     OpenML
# COMPAS           OpenML
# Diabetes         OpenML
# German Credit    https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data
# HELOC            https://community.fico.com/s/explainable-machine-learning-challenge
# HIGGS            OpenML
# Independent      Synthetic dataset from https://proceedings.mlr.press/v196/xenopoulos22a/xenopoulos22a.pdf
# Synthetic        OpenXAI

In [1]:
import time
import numpy as np
import pandas as pd
import zipfile as zf
import nbimporter

import sys

import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
import Taylor_Explainer as texp

# Data preprocessing methods

In [3]:
# each categorical attribute is converted in 0/1 attributes using pandas.get_dummies
# drop_first=True only for attributes with binary values

# receive data as a pandas.DataFrame with categorical attributes
# cat_cols list indicating the columns names from df are numerical

# RETURN one-hot encoded data

def get_dummies_drop_first_only_binary_atts(data, cat_cols):
    df= data.copy()

    for i in cat_cols:
        if len(df.groupby([i]).size())> 2:
            df= pd.get_dummies(df, prefix=[i], columns=[i])
        else:
            df= pd.get_dummies(df, prefix=[i], columns=[i], drop_first=True)
    
    return df

# Adult Income

In [4]:
X = pd.read_csv('data/adult/raw/data.csv')
del X['education']
X.head()

,age,workclass,fnlwgt,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,target
0,25,Private,226802.0,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0.0,0.0,40,United-States,0
1,38,Private,89814.0,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0.0,0.0,50,United-States,0
2,28,Local-gov,336951.0,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0.0,0.0,40,United-States,1
3,44,Private,160323.0,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688.0,0.0,40,United-States,1
4,18,NaN,103497.0,10,Never-married,NaN,Own-child,White,Female,0.0,0.0,30,United-States,0


In [5]:
y = X[['target']]
X = X.drop(['target'], axis=1)
y.head()

,target
0,0
1,0
2,1
3,1
4,0


In [6]:
categor_columns = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
numeric_columns = list(filter(lambda x:x not in categor_columns, X.columns))

numeric_columns

['age',
 'fnlwgt',
 'education-num',
 'capital-gain',
 'capital-loss',
 'hours-per-week']

In [7]:
#Check for null values

texp.check_null_values(X, text_info=True)

There are some missing values in the dataset


In [8]:
X.isna().sum()

age                  0
workclass         2799
fnlwgt               0
education-num        0
marital-status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     857
dtype: int64

In [9]:
# fill missing data
# numerical features are filled with mean values
x= texp.pre_proc_fillna_num_fts(X, numeric_columns, num_type='mean')
# categorical features are filled with mode values
x= texp.pre_proc_fillna_cat_fts(x, categor_columns, cat_type='mode')

In [10]:
# one-hot encoding the categorical features

x_ohe= pd.get_dummies(x, columns=categor_columns)

In [11]:
categor_columns_ohe= list(filter(lambda x:x not in numeric_columns, x_ohe.columns))

categor_columns_ohe

['workclass_Federal-gov',
 'workclass_Local-gov',
 'workclass_Never-worked',
 'workclass_Private',
 'workclass_Self-emp-inc',
 'workclass_Self-emp-not-inc',
 'workclass_State-gov',
 'workclass_Without-pay',
 'marital-status_Divorced',
 'marital-status_Married-AF-spouse',
 'marital-status_Married-civ-spouse',
 'marital-status_Married-spouse-absent',
 'marital-status_Never-married',
 'marital-status_Separated',
 'marital-status_Widowed',
 'occupation_Adm-clerical',
 'occupation_Armed-Forces',
 'occupation_Craft-repair',
 'occupation_Exec-managerial',
 'occupation_Farming-fishing',
 'occupation_Handlers-cleaners',
 'occupation_Machine-op-inspct',
 'occupation_Other-service',
 'occupation_Priv-house-serv',
 'occupation_Prof-specialty',
 'occupation_Protective-serv',
 'occupation_Sales',
 'occupation_Tech-support',
 'occupation_Transport-moving',
 'relationship_Husband',
 'relationship_Not-in-family',
 'relationship_Other-relative',
 'relationship_Own-child',
 'relationship_Unmarried',
 're

In [12]:
# one-hot encoding the categorical features with drop_first only for binary attributes

x_ohe= get_dummies_drop_first_only_binary_atts(x, categor_columns)

In [13]:
categor_columns_ohe= list(filter(lambda x:x not in numeric_columns, x_ohe.columns))

categor_columns_ohe

['workclass_Federal-gov',
 'workclass_Local-gov',
 'workclass_Never-worked',
 'workclass_Private',
 'workclass_Self-emp-inc',
 'workclass_Self-emp-not-inc',
 'workclass_State-gov',
 'workclass_Without-pay',
 'marital-status_Divorced',
 'marital-status_Married-AF-spouse',
 'marital-status_Married-civ-spouse',
 'marital-status_Married-spouse-absent',
 'marital-status_Never-married',
 'marital-status_Separated',
 'marital-status_Widowed',
 'occupation_Adm-clerical',
 'occupation_Armed-Forces',
 'occupation_Craft-repair',
 'occupation_Exec-managerial',
 'occupation_Farming-fishing',
 'occupation_Handlers-cleaners',
 'occupation_Machine-op-inspct',
 'occupation_Other-service',
 'occupation_Priv-house-serv',
 'occupation_Prof-specialty',
 'occupation_Protective-serv',
 'occupation_Sales',
 'occupation_Tech-support',
 'occupation_Transport-moving',
 'relationship_Husband',
 'relationship_Not-in-family',
 'relationship_Other-relative',
 'relationship_Own-child',
 'relationship_Unmarried',
 're

In [14]:
x_ohe.shape

(48842, 88)

In [15]:
# normalize numeric columns of the dataframe with values between 0 and 1

x_ohe= texp.normalize_selected(x_ohe)

In [16]:
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(x_ohe, y, 
                                                                                 train_size=0.80, random_state=1234)

train.head()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,workclass_Federal-gov,workclass_Local-gov,workclass_Never-worked,workclass_Private,...,native-country_Portugal,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia
32721,0.438356,0.173159,0.333333,0.0,0.0,0.397959,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
36170,0.287671,0.176576,0.400000,0.0,0.0,0.316327,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
19838,0.328767,0.047431,0.800000,0.0,0.0,0.500000,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4878,0.493151,0.081197,0.600000,0.0,0.0,0.397959,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
45984,0.232877,0.126473,0.533333,0.0,0.0,0.500000,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [17]:
train.to_csv('./data/adult/processed/X_train.csv', index=False)
test.to_csv('./data/adult/processed/X_test.csv', index=False)
labels_train.to_csv('./data/adult/processed/y_train.csv', index=False)
labels_test.to_csv('./data/adult/processed/y_test.csv', index=False)

# Compas

In [18]:
X = pd.read_csv('data/compas/raw/data.csv')
X.head()

,sex,age,juv_fel_count,juv_misd_count,juv_other_count,priors_count,age_cat_25-45,age_cat_Greaterthan45,age_cat_Lessthan25,race_African-American,race_Caucasian,c_charge_degree_F,c_charge_degree_M,target
0,1,34,0,0,0,0,1,0,0,1,0,1,0,1
1,1,24,0,0,1,4,0,0,1,1,0,1,0,1
2,1,41,0,0,0,14,1,0,0,0,1,1,0,1
3,0,39,0,0,0,0,1,0,0,0,1,0,1,0
4,1,27,0,0,0,0,1,0,0,0,1,1,0,0


In [19]:
y = X[['target']]
X = X.drop(['target'], axis=1)
y.head()

,target
0,1
1,1
2,1
3,0
4,0


In [20]:
categor_columns = []
numeric_columns = list(filter(lambda x:x not in categor_columns, X.columns))

numeric_columns

['sex',
 'age',
 'juv_fel_count',
 'juv_misd_count',
 'juv_other_count',
 'priors_count',
 'age_cat_25-45',
 'age_cat_Greaterthan45',
 'age_cat_Lessthan25',
 'race_African-American',
 'race_Caucasian',
 'c_charge_degree_F',
 'c_charge_degree_M']

In [21]:
#Check for null values

texp.check_null_values(X, text_info=True)
X.isna().sum()

There are no missing values in the dataset


sex                      0
age                      0
juv_fel_count            0
juv_misd_count           0
juv_other_count          0
priors_count             0
age_cat_25-45            0
age_cat_Greaterthan45    0
age_cat_Lessthan25       0
race_African-American    0
race_Caucasian           0
c_charge_degree_F        0
c_charge_degree_M        0
dtype: int64

In [22]:
# normalize numeric columns of the dataframe with values between 0 and 1

x = texp.normalize_selected(X)

In [23]:
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(x, y, 
                                                                                 train_size=0.80,
                                                                                 random_state=1234)

train.head()

,sex,age,juv_fel_count,juv_misd_count,juv_other_count,priors_count,age_cat_25-45,age_cat_Greaterthan45,age_cat_Lessthan25,race_African-American,race_Caucasian,c_charge_degree_F,c_charge_degree_M
3749,0.0,0.112903,0.0,0.0,0.0,0.105263,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1894,0.0,0.225806,0.0,0.0,0.0,0.000000,1.0,0.0,0.0,1.0,0.0,0.0,1.0
2517,1.0,0.241935,0.0,0.0,0.0,0.105263,1.0,0.0,0.0,1.0,0.0,1.0,0.0
4506,1.0,0.225806,0.0,0.0,0.0,0.105263,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1131,1.0,0.580645,0.0,0.0,0.0,0.210526,0.0,1.0,0.0,1.0,0.0,1.0,0.0


In [24]:
train.to_csv('./data/compas/processed/X_train.csv', index=False)
test.to_csv('./data/compas/processed/X_test.csv', index=False)
labels_train.to_csv('./data/compas/processed/y_train.csv', index=False)
labels_test.to_csv('./data/compas/processed/y_test.csv', index=False)

# Diabetes

In [25]:
X = pd.read_csv('data/diabetes/raw/data.csv')
X.head()

,preg,plas,pres,skin,insu,mass,pedi,age,target
0,6.0,148.0,72.0,35.0,0.0,33.6,0.627,50.0,1
1,1.0,85.0,66.0,29.0,0.0,26.6,0.351,31.0,0
2,8.0,183.0,64.0,0.0,0.0,23.3,0.672,32.0,1
3,1.0,89.0,66.0,23.0,94.0,28.1,0.167,21.0,0
4,0.0,137.0,40.0,35.0,168.0,43.1,2.288,33.0,1


In [26]:
y = X[['target']]
X = X.drop(['target'], axis=1)
y.head()

,target
0,1
1,0
2,1
3,0
4,1


In [27]:
categor_columns = []
numeric_columns = list(filter(lambda x:x not in categor_columns, X.columns))

numeric_columns

['preg', 'plas', 'pres', 'skin', 'insu', 'mass', 'pedi', 'age']

In [28]:
#Check for null values

texp.check_null_values(X, text_info=True)
X.isna().sum()

There are no missing values in the dataset


preg    0
plas    0
pres    0
skin    0
insu    0
mass    0
pedi    0
age     0
dtype: int64

In [29]:
# normalize numeric columns of the dataframe with values between 0 and 1

x = texp.normalize_selected(X)

In [30]:
x

,preg,plas,pres,skin,insu,mass,pedi,age
0,0.352941,0.743719,0.590164,0.353535,0.000000,0.500745,0.234415,0.483333
1,0.058824,0.427136,0.540984,0.292929,0.000000,0.396423,0.116567,0.166667
2,0.470588,0.919598,0.524590,0.000000,0.000000,0.347243,0.253629,0.183333
3,0.058824,0.447236,0.540984,0.232323,0.111111,0.418778,0.038002,0.000000
4,0.000000,0.688442,0.327869,0.353535,0.198582,0.642325,0.943638,0.200000
...,...,...,...,...,...,...,...,...
763,0.588235,0.507538,0.622951,0.484848,0.212766,0.490313,0.039710,0.700000
764,0.117647,0.613065,0.573770,0.272727,0.000000,0.548435,0.111870,0.100000
765,0.294118,0.608040,0.590164,0.232323,0.132388,0.390462,0.071307,0.150000
766,0.058824,0.633166,0.491803,0.000000,0.000000,0.448584,0.115713,0.433333


In [31]:
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(x, y, 
                                                                                 train_size=0.80,
                                                                                 random_state=1234)

train.head()

,preg,plas,pres,skin,insu,mass,pedi,age
733,0.117647,0.532663,0.459016,0.272727,0.195035,0.432191,0.148591,0.016667
394,0.235294,0.793970,0.639344,0.000000,0.000000,0.490313,0.309564,0.166667
359,0.058824,0.984925,0.622951,0.363636,0.294326,0.543964,0.340307,0.133333
566,0.058824,0.497487,0.590164,0.303030,0.021277,0.575261,0.142613,0.000000
654,0.058824,0.532663,0.573770,0.282828,0.159574,0.509687,0.027327,0.016667


In [32]:
train.to_csv('./data/diabetes/processed/X_train.csv', index=False)
test.to_csv('./data/diabetes/processed/X_test.csv', index=False)
labels_train.to_csv('./data/diabetes/processed/y_train.csv', index=False)
labels_test.to_csv('./data/diabetes/processed/y_test.csv', index=False)

# German Credit

In [33]:
raw_data_ger= pd.read_csv('data/german/raw/german_credit_data.csv', index_col='ID')

raw_data_ger.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
ID,,,,,,,,,
0,67,male,2,own,NaN,little,1169,6,radio/TV
1,22,female,2,own,little,moderate,5951,48,radio/TV
2,49,male,1,own,little,NaN,2096,12,education
3,45,male,2,free,little,little,7882,42,furniture/equipment
4,53,male,2,free,little,little,4870,24,car


In [34]:
# get the target column (y)

raw_data_ger_target= pd.read_csv('data/german/raw/german.data', delimiter=' ', header=None)

t_pos= raw_data_ger_target.shape[1]

y_ger= raw_data_ger_target.loc[:,raw_data_ger_target.columns[(t_pos-1):t_pos]]
y_ger= y_ger.replace(2, 0)

y_ger.head()

,20
0,1
1,0
2,1
3,1
4,0


In [35]:
categor_columns_ger= ['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']
numeric_columns_ger= list(filter(lambda x:x not in categor_columns_ger, raw_data_ger.columns))

numeric_columns_ger

['Age', 'Job', 'Credit amount', 'Duration']

In [36]:
#Check for null values

texp.check_null_values(raw_data_ger, text_info=True)

There are some missing values in the dataset


In [37]:
# fill missing data
# numerical features are filled with mean values
x_ger= texp.pre_proc_fillna_num_fts(raw_data_ger, numeric_columns_ger, num_type='mean')
# categorical features are filled with mode values
x_ger= texp.pre_proc_fillna_cat_fts(x_ger, categor_columns_ger, cat_type='mode')

In [38]:
# one-hot encoding the categorical features

x_ger_ohe= pd.get_dummies(x_ger, columns=categor_columns_ger)

In [39]:
categor_columns_ger_ohe= list(filter(lambda x:x not in numeric_columns_ger, x_ger_ohe.columns))

categor_columns_ger_ohe

['Sex_female',
 'Sex_male',
 'Housing_free',
 'Housing_own',
 'Housing_rent',
 'Saving accounts_little',
 'Saving accounts_moderate',
 'Saving accounts_quite rich',
 'Saving accounts_rich',
 'Checking account_little',
 'Checking account_moderate',
 'Checking account_rich',
 'Purpose_business',
 'Purpose_car',
 'Purpose_domestic appliances',
 'Purpose_education',
 'Purpose_furniture/equipment',
 'Purpose_radio/TV',
 'Purpose_repairs',
 'Purpose_vacation/others']

In [40]:
# one-hot encoding the categorical features with drop_first only for binary attributes

x_ger_ohe= get_dummies_drop_first_only_binary_atts(x_ger, categor_columns_ger)

In [41]:
categor_columns_ger_ohe= list(filter(lambda x:x not in numeric_columns_ger, x_ger_ohe.columns))

categor_columns_ger_ohe

['Sex_male',
 'Housing_free',
 'Housing_own',
 'Housing_rent',
 'Saving accounts_little',
 'Saving accounts_moderate',
 'Saving accounts_quite rich',
 'Saving accounts_rich',
 'Checking account_little',
 'Checking account_moderate',
 'Checking account_rich',
 'Purpose_business',
 'Purpose_car',
 'Purpose_domestic appliances',
 'Purpose_education',
 'Purpose_furniture/equipment',
 'Purpose_radio/TV',
 'Purpose_repairs',
 'Purpose_vacation/others']

In [42]:
x_ger_ohe.shape

# original data --  9 features
# ohe data      -- 23 features 

# numeric fts   --  4 features
# categorical   --  5 features
# ohe fts       -- 19 features

(1000, 23)

In [43]:
# normalize numeric columns of the dataframe with values between 0 and 1

x_ger_ohe= texp.normalize_selected(x_ger_ohe)

In [44]:
train_ger, test_ger, labels_train_ger, labels_test_ger= sklearn.model_selection.train_test_split(x_ger_ohe,
                                                                                             y_ger,
                                                                                             train_size=0.80,
                                                                                             random_state=1234)

train_ger.head()

,Age,Job,Credit amount,Duration,Sex_male,Housing_free,Housing_own,Housing_rent,Saving accounts_little,Saving accounts_moderate,...,Checking account_moderate,Checking account_rich,Purpose_business,Purpose_car,Purpose_domestic appliances,Purpose_education,Purpose_furniture/equipment,Purpose_radio/TV,Purpose_repairs,Purpose_vacation/others
ID,,,,,,,,,,,,,,,,,,,,,
281,0.553571,0.666667,0.072851,0.117647,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
42,0.446429,0.333333,0.327611,0.205882,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
255,0.142857,0.333333,0.394410,0.823529,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
906,0.089286,0.333333,0.193298,0.250000,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
394,0.214286,1.000000,0.118631,0.073529,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [45]:
labels_train_ger.columns = ['target']
labels_test_ger.columns = ['target']

In [46]:
train_ger.to_csv('./data/german/processed/X_train.csv', index=False)
test_ger.to_csv('./data/german/processed/X_test.csv', index=False)
labels_train_ger.to_csv('./data/german/processed/y_train.csv', index=False)
labels_test_ger.to_csv('./data/german/processed/y_test.csv', index=False)

# HELOC

In [47]:
heloc= pd.read_csv('data/heloc/raw/heloc_dataset_cleaned.csv')

# split synth into features (x) and target (y)
x_heloc= heloc.loc[:,heloc.columns[1:24]]
y_heloc= heloc.loc[:,heloc.columns[0:1]]

x_heloc.head()

,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,MaxDelq2PublicRecLast12M,...,PercentInstallTrades,MSinceMostRecentInqexcl7days,NumInqLast6M,NumInqLast6Mexcl7days,NetFractionRevolvingBurden,NetFractionInstallBurden,NumRevolvingTradesWBalance,NumInstallTradesWBalance,NumBank2NatlTradesWHighUtilization,PercentTradesWBalance
0,55.0,144.0,4,84,20,3,0,83,2.0,3,...,43,0.0,0,0,33.0,?,8.0,1.0,1.0,69.0
1,61.0,58.0,15,41,2,4,4,100,?,0,...,67,0.0,0,0,0.0,?,0.0,?,?,0.0
2,67.0,66.0,5,24,9,0,0,100,?,7,...,44,0.0,4,4,53.0,66.0,4.0,2.0,1.0,86.0
3,66.0,169.0,1,73,28,1,1,93,76.0,6,...,57,0.0,5,4,72.0,83.0,6.0,4.0,3.0,91.0
4,81.0,333.0,27,132,12,0,0,100,?,7,...,25,0.0,1,1,51.0,89.0,3.0,1.0,0.0,80.0


In [48]:
print(np.round(x_heloc.memory_usage().sum() / 10**6, 2), "MB")

1.82 MB


In [49]:
y_heloc.head()

,RiskPerformance
0,Bad
1,Bad
2,Bad
3,Bad
4,Bad


In [50]:
y_heloc= y_heloc.replace(to_replace=['Bad', 'Good'], value=[0, 1])

y_heloc.head()

,RiskPerformance
0,0
1,0
2,0
3,0
4,0


In [51]:
# there are missing values filled with a '?' character. we replaced it with nan values
x_heloc_clean= x_heloc.replace('?', np.nan)
x_heloc_clean.head()

,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,MaxDelq2PublicRecLast12M,...,PercentInstallTrades,MSinceMostRecentInqexcl7days,NumInqLast6M,NumInqLast6Mexcl7days,NetFractionRevolvingBurden,NetFractionInstallBurden,NumRevolvingTradesWBalance,NumInstallTradesWBalance,NumBank2NatlTradesWHighUtilization,PercentTradesWBalance
0,55.0,144.0,4,84,20,3,0,83,2.0,3,...,43,0.0,0,0,33.0,NaN,8.0,1.0,1.0,69.0
1,61.0,58.0,15,41,2,4,4,100,NaN,0,...,67,0.0,0,0,0.0,NaN,0.0,NaN,NaN,0.0
2,67.0,66.0,5,24,9,0,0,100,NaN,7,...,44,0.0,4,4,53.0,66.0,4.0,2.0,1.0,86.0
3,66.0,169.0,1,73,28,1,1,93,76.0,6,...,57,0.0,5,4,72.0,83.0,6.0,4.0,3.0,91.0
4,81.0,333.0,27,132,12,0,0,100,NaN,7,...,25,0.0,1,1,51.0,89.0,3.0,1.0,0.0,80.0


In [52]:
#Check for null values

texp.check_null_values(x_heloc_clean, text_info=True)

There are some missing values in the dataset


In [53]:
categor_columns_heloc= ['MaxDelq2PublicRecLast12M','MaxDelqEver']
numeric_columns_heloc= list(filter(lambda x:x not in categor_columns_heloc, x_heloc_clean.columns))

In [54]:
# fill missing data
# numerical features are filled with mean values
x_heloc_clean= texp.pre_proc_fillna_num_fts((x_heloc_clean.astype(float)), numeric_columns_heloc, 
                                            num_type='mean')
# categorical features are filled with mode values
x_heloc_clean= texp.pre_proc_fillna_cat_fts(x_heloc_clean, categor_columns_heloc, cat_type='mode')

In [55]:
# one-hot encoding the categorical features

x_heloc_ohe= pd.get_dummies(x_heloc_clean, columns=categor_columns_heloc)

In [56]:
categor_columns_heloc_ohe= list(filter(lambda x:x not in numeric_columns_heloc, x_heloc_ohe.columns))

categor_columns_heloc_ohe

['MaxDelq2PublicRecLast12M_0.0',
 'MaxDelq2PublicRecLast12M_1.0',
 'MaxDelq2PublicRecLast12M_2.0',
 'MaxDelq2PublicRecLast12M_3.0',
 'MaxDelq2PublicRecLast12M_4.0',
 'MaxDelq2PublicRecLast12M_5.0',
 'MaxDelq2PublicRecLast12M_6.0',
 'MaxDelq2PublicRecLast12M_7.0',
 'MaxDelq2PublicRecLast12M_9.0',
 'MaxDelqEver_2.0',
 'MaxDelqEver_3.0',
 'MaxDelqEver_4.0',
 'MaxDelqEver_5.0',
 'MaxDelqEver_6.0',
 'MaxDelqEver_7.0',
 'MaxDelqEver_8.0']

In [57]:
x_heloc_ohe.shape

# original data -- 23 features
# ohe data      -- 37 features 

# numeric fts   -- 21 features
# categorical   --  2 features
# ohe fts       -- 16 features

(9871, 37)

In [58]:
# normalize numeric columns of the dataframe with values between 0 and 1

x_heloc_ohe= texp.normalize_selected(x_heloc_ohe)

In [59]:
train_hel, test_hel, labels_train_hel, labels_test_hel= sklearn.model_selection.train_test_split(x_heloc_ohe,
                                                                                             y_heloc,
                                                                                             train_size=0.80,
                                                                                             random_state=1234)

train_hel.head()

,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,NumTotalTrades,...,MaxDelq2PublicRecLast12M_6.0,MaxDelq2PublicRecLast12M_7.0,MaxDelq2PublicRecLast12M_9.0,MaxDelqEver_2.0,MaxDelqEver_3.0,MaxDelqEver_4.0,MaxDelqEver_5.0,MaxDelqEver_6.0,MaxDelqEver_7.0,MaxDelqEver_8.0
720,0.573770,0.072409,0.018277,0.071240,0.139241,0.052632,0.0,0.92,0.325301,0.115385,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1759,0.442623,0.151061,0.002611,0.113456,0.227848,0.000000,0.0,1.00,0.263609,0.192308,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3964,0.524590,0.067416,0.007833,0.044855,0.088608,0.000000,0.0,0.86,0.457831,0.067308,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1786,0.885246,0.159800,0.031332,0.110818,0.329114,0.000000,0.0,1.00,0.263609,0.250000,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
9428,0.377049,0.238452,0.120104,0.221636,0.278481,0.000000,0.0,0.78,0.036145,0.221154,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [60]:
labels_train_hel.columns = ['target']
labels_test_hel.columns = ['target']

In [61]:
train_hel.to_csv('./data/heloc/processed/X_train.csv', index=False)
test_hel.to_csv('./data/heloc/processed/X_test.csv', index=False)
labels_train_hel.to_csv('./data/heloc/processed/y_train.csv', index=False)
labels_test_hel.to_csv('./data/heloc/processed/y_test.csv', index=False)

# HIGGS

In [62]:
ds= zf.ZipFile('data/higgs/raw/higgs_98k.zip')

higgs= pd.read_csv(ds.open('higgs_98k.csv'), low_memory=False)

# split synth into features (x) and target (y)
x_higgs= higgs.loc[:,higgs.columns[1:29]]
y_higgs= higgs.loc[:,higgs.columns[0:1]]

x_higgs.head()

,lepton_pT,lepton_eta,lepton_phi,missing_energy_magnitude,missing_energy_phi,jet1pt,jet1eta,jet1phi,jet1b-tag,jet2pt,...,jet4eta,jet4phi,jet4b-tag,m_jj,m_jjj,m_lv,m_jlv,m_bb,m_wbb,m_wwbb
0,0.907542,0.329147,0.359412,1.497970,-0.313010,1.095531,-0.557525,-1.588230,2.173076,0.812581,...,-1.138930,-0.000819110195152462,0,0.302219897508621,0.833048164844513,0.985699653625488,0.978098392486572,0.779732167720795,0.992355763912201,0.79834258556366
1,0.798835,1.470639,-1.635975,0.453773,0.425629,1.104875,1.282322,1.381664,0.000000,0.851737,...,1.128848,0.900460839271545,0,0.909753262996674,1.10833048820496,0.985692203044891,0.951331257820129,0.803251504898071,0.865924417972565,0.780117571353912
2,1.344385,-0.876626,0.935913,1.992050,0.882454,1.786066,-1.646778,-0.942383,0.000000,2.423265,...,-0.678379,-1.36035633087158,0,0.946652472019196,1.0287036895752,0.998656094074249,0.728280603885651,0.869200229644775,1.02673649787903,0.957903981208801
3,1.105009,0.321356,1.522401,0.882808,-1.205349,0.681466,-1.070464,-0.921871,0.000000,0.800872,...,-0.373566,0.113040611147881,0,0.755856454372406,1.36105704307556,0.986609697341919,0.838084638118744,1.13329517841339,0.872244894504547,0.808486521244049
4,1.595839,-0.607811,0.007075,1.818450,-0.111906,0.847550,-0.566437,1.581239,2.173076,0.755421,...,-0.654227,-1.27434492111206,3.10196137428284,0.823760569095612,0.938191413879395,0.971758186817169,0.789176344871521,0.430553287267685,0.961356937885284,0.957817912101746


In [63]:
print(np.round(x_higgs.memory_usage().sum() / 10**6, 2), "MB")

21.96 MB


In [64]:
y_higgs.head()

,class
0,1
1,1
2,0
3,1
4,0


In [65]:
# Check for null values

texp.check_null_values(x_higgs, text_info=True)

# All the values are numeric in the HIGGS dataset

There are no missing values in the dataset


In [66]:
x_higgs.isin(['?']).sum(axis=0)

lepton_pT                   0
lepton_eta                  0
lepton_phi                  0
missing_energy_magnitude    0
missing_energy_phi          0
jet1pt                      0
jet1eta                     0
jet1phi                     0
jet1b-tag                   0
jet2pt                      0
jet2eta                     0
jet2phi                     0
jet2b-tag                   0
jet3pt                      0
jet3eta                     0
jet3phi                     0
jet3b-tag                   0
jet4pt                      0
jet4eta                     0
jet4phi                     1
jet4b-tag                   1
m_jj                        1
m_jjj                       1
m_lv                        1
m_jlv                       1
m_bb                        1
m_wbb                       1
m_wwbb                      1
dtype: int64

In [67]:
x_higgs= x_higgs.replace('?', np.nan)

x_higgs= x_higgs.astype(float)

# fill missing (numeric) data with mean values
x_higgs= texp.pre_proc_fillna_num_fts(x_higgs, x_higgs.columns, num_type='mean')

In [68]:
# normalize numeric columns of the dataframe with values between 0 and 1

x_higgs= texp.normalize_selected(x_higgs)

In [69]:
train_hig, test_hig, labels_train_hig, labels_test_hig= sklearn.model_selection.train_test_split(x_higgs,
                                                                                             y_higgs,
                                                                                             train_size=0.80,
                                                                                             random_state=1234)

train_hig.head()

,lepton_pT,lepton_eta,lepton_phi,missing_energy_magnitude,missing_energy_phi,jet1pt,jet1eta,jet1phi,jet1b-tag,jet2pt,...,jet4eta,jet4phi,jet4b-tag,m_jj,m_jjj,m_lv,m_jlv,m_bb,m_wbb,m_wwbb
40951,0.022891,0.759352,0.876317,0.125775,0.364631,0.100052,0.691679,0.540606,1.0,0.027402,...,0.758086,0.287798,0.0,0.018702,0.056143,0.192729,0.067751,0.139907,0.112800,0.086516
76614,0.073727,0.395679,0.671134,0.159534,0.417735,0.111667,0.538436,0.925663,0.0,0.082393,...,0.108536,0.276655,0.0,0.091208,0.082455,0.199951,0.073763,0.085361,0.096756,0.100368
97143,0.023231,0.716143,0.401294,0.127702,0.715179,0.055456,0.571619,0.800706,1.0,0.085941,...,0.727076,0.213142,0.0,0.030815,0.035638,0.193884,0.105637,0.077562,0.076504,0.070331
57634,0.424256,0.546509,0.804049,0.080008,0.627251,0.166157,0.640654,0.175576,0.0,0.244317,...,0.591030,0.476586,0.0,0.044032,0.087570,0.215782,0.060640,0.161801,0.162391,0.176001
16835,0.201448,0.663533,0.685778,0.074547,0.956967,0.172640,0.591796,0.181306,1.0,0.015514,...,0.236746,0.556842,0.0,0.035411,0.110135,0.213415,0.061279,0.064803,0.089685,0.091670


In [70]:
labels_train_hig.columns = ['target']
labels_test_hig.columns = ['target']

In [71]:
train_hig.to_csv('./data/higgs/processed/X_train.csv', index=False)
test_hig.to_csv('./data/higgs/processed/X_test.csv', index=False)
labels_train_hig.to_csv('./data/higgs/processed/y_train.csv', index=False)
labels_test_hig.to_csv('./data/higgs/processed/y_test.csv', index=False)

# Independent

In [72]:
X = pd.read_csv('data/independent/raw/data.csv')
X.head()

,x1,x2,x3,x4,x5,x6,y
0,0.019424,-0.435155,-0.116725,0.376715,0.069791,-0.255029,0
1,-0.727128,-1.000000,0.707138,-0.714598,0.656234,0.084969,0
2,0.062495,-0.159553,-0.175131,0.139016,-0.213702,0.469571,0
3,-0.245510,-0.297415,0.176057,0.082185,-0.290922,0.868176,0
4,-0.041190,0.255027,-0.357788,-0.254220,0.303073,-0.554674,0


In [73]:
y = X[['y']]
X = X.drop(['y'], axis=1)
y.head()

,y
0,0
1,0
2,0
3,0
4,0


In [74]:
categor_columns = []
numeric_columns = list(filter(lambda x:x not in categor_columns, X.columns))

numeric_columns

['x1', 'x2', 'x3', 'x4', 'x5', 'x6']

In [75]:
#Check for null values

texp.check_null_values(X, text_info=True)
X.isna().sum()

There are no missing values in the dataset


x1    0
x2    0
x3    0
x4    0
x5    0
x6    0
dtype: int64

In [76]:
# normalize numeric columns of the dataframe with values between 0 and 1

x = texp.normalize_selected(X)

In [77]:
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(x, y, 
                                                                                 train_size=0.80, random_state=1234)

train.head()

,x1,x2,x3,x4,x5,x6
145,0.228416,0.159086,0.357537,0.551525,0.294385,0.440340
284,0.301897,0.632029,0.483163,0.399265,0.631941,0.811079
52,0.647000,0.426842,0.469701,0.312124,0.617747,0.776539
272,0.585683,0.476598,0.292525,0.645296,0.404119,0.555664
277,0.476208,0.582202,0.157767,0.566446,0.677517,0.232993


In [78]:
labels_train.columns = ['target']
labels_test.columns = ['target']

In [79]:
train.to_csv('./data/independent/processed/X_train.csv', index=False)
test.to_csv('./data/independent/processed/X_test.csv', index=False)
labels_train.to_csv('./data/independent/processed/y_train.csv', index=False)
labels_test.to_csv('./data/independent/processed/y_test.csv', index=False)

# Synthetic

In [80]:
synth_ox= pd.read_csv('data/synth_OX_20/raw/synth_OX_20.csv')

# split synth into features (x) and target (y)
ox_inputs= synth_ox.loc[:,synth_ox.columns[0:20]]
ox_labels= synth_ox.loc[:,synth_ox.columns[20:21]]

ox_inputs.head()

,ft_1,ft_2,ft_3,ft_4,ft_5,ft_6,ft_7,ft_8,ft_9,ft_10,ft_11,ft_12,ft_13,ft_14,ft_15,ft_16,ft_17,ft_18,ft_19,ft_20
0,0.325761,0.472693,0.385410,0.312884,0.271236,0.207762,0.620768,0.083032,0.380621,0.385330,0.661285,0.607623,0.369915,0.662277,0.266994,0.327500,0.532432,0.394437,0.340003,0.429918
1,0.391322,0.439526,0.399563,0.394650,0.292309,0.425911,0.405543,0.767337,0.396363,0.373539,0.434264,0.481565,0.244531,0.238732,0.337101,0.624010,0.301257,0.388973,0.563497,0.570738
2,0.381488,0.723214,0.272272,0.415180,0.283829,0.428776,0.359133,0.394703,0.170128,0.264208,0.340062,0.436833,0.357293,0.505398,0.311589,0.651714,0.489318,0.498270,0.449700,0.531800
3,0.306399,0.198559,0.360769,0.301099,0.801582,0.207521,0.348990,0.240065,0.317928,0.134672,0.538223,0.613235,0.373322,0.494310,0.436847,0.372760,0.671813,0.506265,0.534691,0.509233
4,0.298658,0.286087,0.301725,0.460478,0.515515,0.264769,0.375526,0.297023,0.271326,0.711050,0.506832,0.484622,0.405023,0.355244,0.330762,0.502291,0.423997,0.393348,0.255827,0.477407


In [81]:
print(np.round(ox_inputs.memory_usage().sum() / 10**6, 2), "MB")

0.16 MB


In [82]:
# split ox_inputs and df_labels into train (80%) and test (20%) datasets

train_ox, test_ox, labels_train_ox, labels_test_ox= sklearn.model_selection.train_test_split(ox_inputs,
                                                                                             ox_labels,
                                                                                             train_size=0.80,
                                                                                             random_state=1234)
train_ox.head()

,ft_1,ft_2,ft_3,ft_4,ft_5,ft_6,ft_7,ft_8,ft_9,ft_10,ft_11,ft_12,ft_13,ft_14,ft_15,ft_16,ft_17,ft_18,ft_19,ft_20
281,0.705423,0.250877,0.393413,0.353529,0.328286,0.374804,0.312217,0.130608,0.281738,0.368790,0.758551,0.608015,0.440911,0.481930,0.263193,0.348485,0.127619,0.420548,0.382225,0.347964
42,0.119699,0.665472,0.235255,0.404961,0.292727,0.321720,0.246230,0.324422,0.360535,0.228708,0.630680,0.377279,0.328190,0.445910,0.344165,0.403461,0.579152,0.558943,0.448307,0.455760
255,0.226558,0.670225,0.344595,0.497451,0.298231,0.316184,0.430817,0.313977,0.414127,0.390934,0.637601,0.566323,0.420372,0.513464,0.609983,0.332080,0.327387,0.239984,0.463658,0.595457
906,0.229718,0.291506,0.394624,0.390758,0.310606,0.680242,0.283559,0.345412,0.421726,0.394265,0.589889,0.494661,0.640523,0.478399,0.437791,0.589921,0.648241,0.700112,0.269150,0.385109
394,0.373117,0.249822,0.259671,0.329655,0.307589,0.657956,0.408162,0.318289,0.333588,0.428981,0.631567,0.439209,0.392151,0.282901,0.562405,0.525431,0.416705,0.713325,0.586582,0.639019


In [84]:
train_ox.to_csv('./data/synth_OX_20/processed/X_train.csv', index=False)
test_ox.to_csv('./data/synth_OX_20/processed/X_test.csv', index=False)
labels_train_ox.to_csv('./data/synth_OX_20/processed/y_train.csv', index=False)
labels_test_ox.to_csv('./data/synth_OX_20/processed/y_test.csv', index=False)